``AAPred.score_to_group()`` is a stateless helper that maps continuous prediction scores to an **ordered categorical** of named confidence bands. ``thresholds`` delimit the bands and ``labels`` names them from the lowest to the highest scores; each threshold is an *inclusive lower bound*, so a score equal to a threshold falls in the band above it. It is the single source of truth for the same bands drawn by :meth:`AAPredPlot.predict_group` (``band=True``), so a table and its plot always agree.

In [1]:
import aaanalysis as aa
aa.options["verbose"] = False  # Disable verbosity

# DOM_GSEC example dataset + its feature set (see [Breimann25]_)
df_seq = aa.load_dataset(name="DOM_GSEC")
labels = df_seq["label"].to_list()
df_feat = aa.load_features(name="DOM_GSEC").head(20)

# Build the CPP feature matrix and cross-validated out-of-fold scores
sf = aa.SequenceFeature()
df_parts = sf.get_df_parts(df_seq=df_seq)
X = sf.feature_matrix(features=df_feat["feature"], df_parts=df_parts)
aap = aa.AAPred(models=["rf", "extra_trees"], random_state=42)
df_pred = aap.predict_oof(X, labels)

/Users/stephanbreimann/Programming/1Packages/wt-408-score-to-group/aaanalysis/feature_engineering/_backend/cpp_run.py:164: UserWarning: CPP is using the Python kernel fallback — the compiled Cython extension is not available in this install. Output is bit-exact with the Cython path but ~2x slower. Reinstall via `pip install --force-reinstall aaanalysis` to fetch a prebuilt wheel.
  warnings.warn(


Pass the ``scores`` together with ``thresholds`` and one more ``labels`` name than thresholds. Here percent scores (``0-100``) are split into three bands; the returned series is row-aligned with the scores and its categories keep the low-to-high order:

In [2]:
scores = df_pred["score"] * 100  # percent scale
groups = aa.AAPred.score_to_group(scores, thresholds=[50, 80], labels=["low", "medium", "high"])
df_groups = df_seq[["entry"]].assign(score=scores.round(1).values, group=groups.values)
aa.display_df(df_groups, n_rows=10, show_shape=True)

DataFrame shape: (126, 3)


,entry,score,group
1,P05067,80.000000,high
2,P14925,87.000000,high
3,P70180,89.500000,high
4,Q03157,93.500000,high
5,Q06481,98.500000,high
6,P35613,87.000000,high
7,P35070,7.500000,low
8,P09803,94.000000,high
9,P19022,96.000000,high
10,P16070,82.500000,high


``score_range`` selects the numeric scale that ``thresholds`` must lie within: ``'percent'`` (``[0, 100]``, the default) or ``'proba'`` (``[0, 1]``). Thresholds outside the range raise, so probabilities and percentages can't be silently mixed. Scoring the raw ``[0, 1]`` probabilities on the ``'proba'`` scale gives the same banding:

In [3]:
groups_proba = aa.AAPred.score_to_group(df_pred["score"], thresholds=[0.5, 0.8],
                                        labels=["low", "medium", "high"], score_range="proba")
df_counts = groups_proba.value_counts(sort=False).rename_axis("group").reset_index(name="n_proteins")
aa.display_df(df_counts, n_rows=10, show_shape=True)

DataFrame shape: (3, 2)


,group,n_proteins
1,low,61
2,medium,35
3,high,30
